Import access token from hugging face.

In [3]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

Download dataset Kazsandra. Divide data by train, test and val


In [6]:
from datasets import load_dataset
import pandas as pd
ds = load_dataset("issai/kazsandra", "polarity_classification")
train = ds["train"].to_pandas()
test = ds["test"].to_pandas()
val = ds["validation"].to_pandas()
print(len(train), len(test), len(val))
print(train.label.value_counts(normalize=True))
train.head()



134368 16797 16796
label
1    0.821751
0    0.178249
Name: proportion, dtype: float64


,custom_id,text,text_cleaned,label,domain
0,pla015439pla,Өтте күшті,өтте күшті,1,appstore
1,pla083193pla,Мәбазар жок .Оте керемет тамаша керемет,мәбазар жок оте керемет тамаша керемет,1,appstore
2,pla113624pla,"Кушти , дал тура айтады 👍👍👍👍",кушти дал тура айтады,1,appstore
3,pla029825pla,Реклама коп,реклама коп,0,appstore
4,pla002604pla,5-баға беремін,5 баға беремін,1,appstore


Compare text and text_cleaned (need to choose thr cloum)

In [8]:
pd.set_option("display.max_colwidth", 200)
print(train[["text", "text_cleaned"]].sample(8, random_state=0))
print("Coincide:", (train.text == train.text_cleaned).mean())

                                                                                                                 text                                                                                     text_cleaned
129996                                           Барлығы түсініксіз неге еговка тіркелсе де баланың даныйы шыкпайд...                                барлығы түсініксіз неге еговка тіркелсе де баланың даныйы шыкпайд
5517                                                        Өте әдемі, іші кең, сапалы, қыздарыма ұнады. Қымбат емес.                                             өте әдемі іші кең сапалы қыздарыма ұнады қымбат емес
107495               Қызықты ойын, бірақ үйлер салғанда кірпіштер тез таусылады.Сонын салдарынан үйлерді тұрғызу кйын  қызықты ойын бірақ үйлер салғанда кірпіштер тез таусылады сонын салдарынан үйлерді тұрғызу кйын
50515                                                                                        Оте жаксы Отбасылык банк                       

Finding duplicates in train and test and remove it.

In [9]:
TEXT = "text_cleaned"

def norm(s): return " ".join(str(s).lower().split())

for df in (train, val, test):
    df["key"] = df[TEXT].map(norm)

dup_mask = train["key"].isin(set(test["key"]))
n_dup = dup_mask.sum()
print(f"Duplicates train↔test: {n_dup} ({n_dup/len(train):.2%} train)")
print("Duplicates val↔test:", val["key"].isin(set(test["key"])).sum())
print("Duplicates in train:", train["key"].duplicated().sum())

train = train[~dup_mask].reset_index(drop=True)
print("train after clean:", len(train))

Duplicates train↔test: 637 (0.47% train)
Duplicates val↔test: 73
Duplicates in train: 2216
train after clean: 133731


Length and limit

In [10]:
for df in (train, val, test):
    df["n_words"] = df[TEXT].str.split().str.len()

threshold = train["n_words"].median()
print("Порог (медиана train):", threshold)

for df in (train, val, test):
    df["group"] = (df["n_words"] > threshold).map({False: "short", True: "long"})

print(test.group.value_counts())
print(test.groupby("group").label.value_counts(normalize=True))

Порог (медиана train): 6.0
group
short    9275
long     7522
Name: count, dtype: int64
group  label
long   1        0.822521
       0        0.177479
short  1        0.821240
       0        0.178760
Name: proportion, dtype: float64


Delete collision val<->test and save data


In [13]:
n_val_dup = val["key"].isin(set(test["key"])).sum()
val = val[~val["key"].isin(set(test["key"]))].reset_index(drop=True)
print("Удалено из val:", n_val_dup, "-> val:", len(val))


conf = train.groupby("key")["label"].nunique()
print("Текстов с конфликтующими метками в train:", (conf > 1).sum())
import os
os.makedirs("data", exist_ok=True)
cols = ["custom_id", "text_cleaned", "label", "domain", "n_words", "group"]
train[cols].to_parquet("data/train.parquet"); val[cols].to_parquet("data/val.parquet"); test[cols].to_parquet("data/test.parquet")

Удалено из val: 0 -> val: 16723
Текстов с конфликтующими метками в train: 1072


baseline TF-IDF + logic regression

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import time

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5),
                      sublinear_tf=True, min_df=3, max_features=300_000)

t = time.time()
X_train = vec.fit_transform(train[TEXT])   # только train
X_val   = vec.transform(val[TEXT])
X_test  = vec.transform(test[TEXT])
print("Признаков:", X_train.shape[1], f"{time.time()-t:.0f} c")

best = None
for C in [0.3, 1, 3, 10]:
    clf = LogisticRegression(C=C, class_weight="balanced", max_iter=1000)
    clf.fit(X_train, train["label"])
    f1 = f1_score(val["label"], clf.predict(X_val), average="macro")
    print(f"C={C}: val macro-F1 = {f1:.4f}")
    if best is None or f1 > best[0]:
        best = (f1, C, clf)

_, best_C, lr = best
print("Лучший C:", best_C)

Признаков: 127494 37 c
C=0.3: val macro-F1 = 0.7484
C=1: val macro-F1 = 0.7518
C=3: val macro-F1 = 0.7531
C=10: val macro-F1 = 0.7459
Лучший C: 3


evaluation by length group

In [16]:
test["pred_lr"] = lr.predict(X_test)

def report(df, pred_col):
    out = {"all": f1_score(df["label"], df[pred_col], average="macro")}
    for g in ["short", "long"]:
        d = df[df["group"] == g]
        out[g] = f1_score(d["label"], d[pred_col], average="macro")
    return out

print(report(test, "pred_lr"))
print(test["group"].value_counts().to_dict(), "порог:", threshold)
test.to_parquet("data/test_with_lr_preds.parquet")

{'all': 0.7629741619525152, 'short': 0.7411302851552675, 'long': 0.790223834831709}
{'short': 9275, 'long': 7522} порог: 6.0
